# Notebook 04 — Model Evaluation

**Project:** Blood Donation Prediction System  
**Authors:** Mihret Alemayehu, Abebech Nega  
**Institution:** Debre Berhan University  
**Instructor:** Petros Abebe  

---

## Objective
This notebook is **fully self-contained**. It rebuilds the complete pipeline
(load → clean → engineer features → train) and then produces a thorough evaluation:

| Section | Content |
|---------|--------|
| 1 | Imports, paths, folder setup |
| 2 | Load raw data, clean, impute, engineer features |
| 3 | Train Random Forest Classifier |
| 4 | Evaluation metrics summary |
| 5 | Confusion matrix (dark theme) |
| 6 | ROC curve with AUC |
| 7 | Precision-Recall curve |
| 8 | Feature importances (rainbow bars) |
| 9 | Full classification report |
| 10 | Save results & HTML report |

> **Tip:** Use *Kernel → Restart & Run All* to regenerate every figure from scratch.


In [ ]:
# ─── Cell 1: ALL imports, magic, paths, folder setup ─────────────────────────
%matplotlib inline

import json
import os
import pickle
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')

# ── Relative paths that work in VS Code (notebook is in notebooks/) ───────────
NB_DIR      = os.path.dirname(os.path.abspath('__file__'))
ROOT        = os.path.abspath(os.path.join(NB_DIR, '..'))
RAW_CSV     = os.path.join(ROOT, 'data', 'raw', 'blood_donation.csv')
PROC_CSV    = os.path.join(ROOT, 'data', 'processed', 'blood_donation_clean.csv')
MODELS_DIR  = os.path.join(ROOT, 'models')
FIGURES_DIR = os.path.join(ROOT, 'reports', 'figures')
EVAL_DIR    = os.path.join(ROOT, 'reports', 'Evaluation results')

os.makedirs(os.path.join(ROOT, 'data', 'processed'), exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,    exist_ok=True)

TARGET = 'Target'

# ── Beautiful chart style ────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.titlepad'     : 14,
    'axes.labelsize'    : 11,
    'figure.facecolor'  : '#FAFBFC',
    'axes.facecolor'    : '#FAFBFC',
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'grid.linestyle'    : '--',
})
sns.set_theme(style='whitegrid', font_scale=1.1)

RED, BLUE, PURPLE, GREEN, AMBER = '#E74C3C', '#3498DB', '#9B59B6', '#2ECC71', '#F39C12'

def save_fig(fig, filename):
    """Save figure to reports/figures/ at high DPI, then display inline."""
    path = os.path.join(FIGURES_DIR, filename)
    fig.savefig(path, dpi=160, bbox_inches='tight', facecolor=fig.get_facecolor())
    print(f'  Saved → reports/figures/{filename}')

print('All libraries imported successfully.')
print(f'Root           : {ROOT}')
print(f'Figures folder : {FIGURES_DIR}')


---
## Section 2 — Load, Clean & Engineer Features

We run the complete preprocessing pipeline inline:
1. Load raw CSV
2. Standardise missing value markers → NaN
3. Remove duplicates
4. KNN Imputation (k=5) on numeric columns
5. Engineer three new features
6. Save processed CSV to `../data/processed/`


In [ ]:
# ─── Cell 2: Load raw data ────────────────────────────────────────────────────
df = pd.read_csv(RAW_CSV)
print(f'Raw shape: {df.shape}')
df.head(3)


In [ ]:
# ─── Cell 3: Standardise NaN markers + drop duplicates ───────────────────────
df = df.replace(['', ' ', 'NA', 'N/A', 'nan', 'NaN', '--', '-', '?'], np.nan)
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Rows before duplicate removal : {before:,}')
print(f'Rows after  duplicate removal : {len(df):,}  (removed {before - len(df)})')
print(f'Missing values per column:')
print(df.isnull().sum().to_string())


In [ ]:
# ─── Cell 4: KNN Imputation (k=5) on numeric columns ─────────────────────────
y = df[TARGET].copy()
X_raw = df.drop(columns=[TARGET])

# Label-encode categorical columns (KNN needs numeric input)
label_encoders = {}
for col in X_raw.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    X_raw[col] = X_raw[col].fillna('Unknown')
    X_raw[col] = le.fit_transform(X_raw[col].astype(str))
    label_encoders[col] = le

missing_before = X_raw.isnull().sum().sum()
knn = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(knn.fit_transform(X_raw), columns=X_raw.columns)
print(f'Missing cells before KNN : {missing_before}')
print(f'Missing cells after  KNN : {X_imputed.isnull().sum().sum()}')


In [ ]:
# ─── Cell 5: Feature engineering + save processed CSV ────────────────────────
if 'Frequency' in X_imputed.columns and 'Time' in X_imputed.columns:
    X_imputed['donation_density']    = X_imputed['Frequency'] / (X_imputed['Time'] + 1e-6)
    X_imputed['recency_score']       = 1.0 / (X_imputed['Recency'] + 1)
    X_imputed['high_frequency_flag'] = (
        X_imputed['Frequency'] > X_imputed['Frequency'].median()
    ).astype(int)
    print('Engineered features: donation_density, recency_score, high_frequency_flag')

df_clean = pd.concat([X_imputed, y.reset_index(drop=True)], axis=1)
df_clean.to_csv(PROC_CSV, index=False)
print(f'Processed dataset shape : {df_clean.shape}')
print(f'Saved → {PROC_CSV}')
df_clean.head(3)


---
## Section 3 — Train the Random Forest Classifier

We use an **80/20 stratified train-test split** and then run **5-fold cross-validation**
to confirm the model generalizes well before evaluating on the held-out test set.


In [ ]:
# ─── Cell 6: Scale features + train/test split ───────────────────────────────
y_all  = df_clean[TARGET]
X_all  = df_clean.drop(columns=[TARGET])

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_all), columns=X_all.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_all, test_size=0.20, random_state=42, stratify=y_all
)
print(f'Train samples : {len(X_train):,}')
print(f'Test  samples : {len(X_test):,}')
print(f'Features      : {X_all.shape[1]}')


In [ ]:
# ─── Cell 7: Train Random Forest + 5-fold CV ─────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
)
rf.fit(X_train, y_train)
print(f'Training accuracy : {rf.score(X_train, y_train):.4f}')

cv_scores = cross_val_score(rf, X_scaled, y_all, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-Fold CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Save model + scaler for the Streamlit app
with open(os.path.join(MODELS_DIR, 'model.pkl'), 'wb') as f:
    pickle.dump(rf, f)
with open(os.path.join(MODELS_DIR, 'preprocessing.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print('Models saved to models/')


---
## Section 4 — Evaluation Metrics Summary

We compute all key metrics on the **held-out test set** (20% of data, never seen during training).


In [ ]:
# ─── Cell 8: Compute all metrics ─────────────────────────────────────────────
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)

metrics_df = pd.DataFrame({
    'Metric'         : ['Accuracy', 'Precision', 'Recall', 'F1 Score',
                        'ROC-AUC', 'CV Mean (5-fold)', 'CV Std'],
    'Score'          : [f'{acc:.4f}', f'{prec:.4f}', f'{rec:.4f}', f'{f1:.4f}',
                        f'{auc:.4f}', f'{cv_scores.mean():.4f}', f'{cv_scores.std():.4f}'],
    'Interpretation' : [
        'Overall correct predictions',
        'Of predicted donors, how many will actually donate',
        'Of actual donors, how many were correctly identified',
        'Harmonic mean of Precision and Recall',
        'Class discrimination ability (1.0 = perfect)',
        'Generalization accuracy across folds',
        'Stability of cross-validated accuracy',
    ],
})
print(f'Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')
print()
metrics_df


---
## Section 5 — Confusion Matrix

The confusion matrix shows exact counts for True Negatives (TN), False Positives (FP),
False Negatives (FN), and True Positives (TP).


In [ ]:
# ─── Cell 9: Confusion matrix (dark theme) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5.2))
fig.patch.set_facecolor('#0F172A')
ax.set_facecolor('#0F172A')

cmap_cm = LinearSegmentedColormap.from_list(
    'dark_blues', ['#1E293B', '#1D4ED8', '#60A5FA'], N=256)
ax.imshow(cm, cmap=cmap_cm, aspect='auto')

ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Won't\nDonate", 'Will\nDonate'], fontsize=12, color='white')
ax.set_yticklabels(["Won't\nDonate", 'Will\nDonate'], fontsize=12, color='white')
ax.set_xlabel('Predicted Label', fontsize=12, color='white')
ax.set_ylabel('True Label',      fontsize=12, color='white')
ax.tick_params(colors='white')

cell_labels = [['TN', 'FP'], ['FN', 'TP']]
for r in range(2):
    for c in range(2):
        ax.text(c, r, f"{cell_labels[r][c]}\n{cm[r,c]:,}",
                ha='center', va='center',
                fontsize=18, fontweight='bold', color='white')

ax.set_title('Confusion Matrix — Random Forest',
             fontsize=14, fontweight='bold', color='white', pad=16)
for sp in ax.spines.values():
    sp.set_visible(False)

save_fig(fig, 'confusion_matrix.png')
plt.show()
plt.close(fig)


---
## Section 6 — ROC Curve

The **ROC curve** plots True Positive Rate vs. False Positive Rate at every threshold.
The **AUC** (area under the curve) summarizes discriminative power — closer to 1.0 is better.
A random classifier scores AUC = 0.50 (dashed baseline).


In [ ]:
# ─── Cell 10: ROC curve (gradient fill + large AUC watermark) ────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color=BLUE, lw=2.8,
        label=f'Random Forest  (AUC = {auc:.3f})', zorder=4)
ax.fill_between(fpr, tpr, alpha=0.18, color=BLUE)
ax.plot([0, 1], [0, 1], 'k--', lw=1.4,
        label='Random Classifier  (AUC = 0.500)')

# Large faded AUC watermark in the chart
ax.text(0.52, 0.08, f'AUC = {auc:.4f}',
        fontsize=24, fontweight='bold', color=BLUE, alpha=0.30,
        transform=ax.transAxes)

ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.05)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate  (Recall)')
ax.set_title('ROC Curve — Random Forest Classifier')
ax.legend(loc='lower right', fontsize=11)

save_fig(fig, 'roc_curve.png')
plt.show()
plt.close(fig)


---
## Section 7 — Precision-Recall Curve

The **Precision-Recall curve** is especially informative for imbalanced datasets.
It shows the trade-off between precision and recall at different decision thresholds.
The dashed line shows the **random baseline** (class frequency).


In [ ]:
# ─── Cell 11: Precision-Recall curve ─────────────────────────────────────────
pr_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rec_vals, pr_vals, color=PURPLE, lw=2.5, label='Random Forest')
ax.fill_between(rec_vals, pr_vals, alpha=0.15, color=PURPLE)
ax.axhline(y=y_test.mean(), color='#94A3B8', linestyle='--', lw=1.4,
           label=f'Baseline ({y_test.mean():.2f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Random Forest')
ax.legend(fontsize=11)

save_fig(fig, 'precision_recall_curve.png')
plt.show()
plt.close(fig)


---
## Section 8 — Feature Importances

Random Forest computes **Gini importance** for each feature — the average reduction in
impurity (node splitting gain) across all 100 trees. Higher = more predictive power.


In [ ]:
# ─── Cell 12: Feature importance (rainbow gradient bars) ─────────────────────
fi_df = pd.DataFrame({
    'feature'   : X_all.columns,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('Top 10 features by Gini importance:')
print(fi_df.head(10).to_string(index=False))


In [ ]:
# ─── Cell 13: Feature importance chart ───────────────────────────────────────
top_fi = fi_df.sort_values('importance')          # ascending for horizontal bar

cmap_fi = plt.cm.RdYlGn
fi_colors = [cmap_fi(v / top_fi['importance'].max()) for v in top_fi['importance']]

fig, ax = plt.subplots(figsize=(9, max(4.5, len(top_fi) * 0.42)))
bars = ax.barh(top_fi['feature'], top_fi['importance'],
               color=fi_colors, edgecolor='white', linewidth=0.8, height=0.68)

for bar, val in zip(bars, top_fi['importance']):
    ax.text(
        bar.get_width() + top_fi['importance'].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.4f}', va='center', ha='left', fontsize=9, fontweight='bold'
    )

ax.set_xlabel('Gini Importance')
ax.set_title('Feature Importances — Random Forest Classifier')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.set_xlim(0, top_fi['importance'].max() * 1.18)

sm = plt.cm.ScalarMappable(
    cmap=cmap_fi, norm=plt.Normalize(0, top_fi['importance'].max()))
fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02, label='Importance')

save_fig(fig, 'feature_importance.png')
plt.show()
plt.close(fig)


---
## Section 9 — Full Classification Report

The classification report shows per-class Precision, Recall, F1, and Support.


In [ ]:
# ─── Cell 14: Classification report ─────────────────────────────────────────
report_str = classification_report(
    y_test, y_pred,
    target_names=["Won't Donate", 'Will Donate']
)
print('Classification Report:')
print('=' * 55)
print(report_str)


---
## Section 10 — Save Model Metadata & HTML Evaluation Report

We save a `model_metadata.json` for the Streamlit app and a beautiful HTML report
for sharing with the instructor.


In [ ]:
# ─── Cell 15: Save model metadata JSON ───────────────────────────────────────
metadata = {
    'project'        : 'Blood Donation Prediction System',
    'authors'        : ['Mihret Alemayehu', 'Abebech Nega'],
    'institution'    : 'Debre Berhan University',
    'instructor'     : 'Petros Abebe',
    'algorithm'      : 'Random Forest Classifier',
    'hyperparameters': {'n_estimators': 100, 'random_state': 42,
                        'class_weight': 'balanced', 'n_jobs': -1},
    'split'          : {'test_size': 0.20, 'train_size': 0.80,
                        'stratified': True, 'random_state': 42},
    'train_samples'  : int(len(X_train)),
    'test_samples'   : int(len(X_test)),
    'features'       : X_all.columns.tolist(),
    'n_features'     : int(X_all.shape[1]),
    'evaluation'     : {
        'accuracy'        : round(float(acc),  4),
        'precision'       : round(float(prec), 4),
        'recall'          : round(float(rec),  4),
        'f1_score'        : round(float(f1),   4),
        'roc_auc'         : round(float(auc),  4),
        'cv_mean_accuracy': round(float(cv_scores.mean()), 4),
        'cv_std_accuracy' : round(float(cv_scores.std()),  4),
    },
    'imputation'     : {'method': 'KNN', 'n_neighbors': 5},
    'scaling'        : 'StandardScaler',
    'saved_at'       : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}
meta_path = os.path.join(MODELS_DIR, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f'Metadata saved → {meta_path}')


In [ ]:
# ─── Cell 16: Generate beautiful HTML evaluation report ──────────────────────
fi_rows = ''.join(
    f'<tr><td>{i+1}</td><td>{row["feature"]}</td>'
    f'<td>{row["importance"]:.4f}</td></tr>'
    for i, row in fi_df.head(10).iterrows()
)

html = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
  <meta charset=\"UTF-8\">
  <meta name=\"viewport\" content=\"width=device-width,initial-scale=1.0\">
  <title>Evaluation Report — Blood Donation Prediction</title>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap');
    *,*::before,*::after{{box-sizing:border-box;margin:0;padding:0;}}
    body{{font-family:'Inter',sans-serif;background:linear-gradient(135deg,#0f0c29,#302b63,#24243e);
         min-height:100vh;color:#E2E8F0;padding:32px 16px 64px;}}
    .wrap{{max-width:960px;margin:0 auto;}}
    .hero{{background:linear-gradient(135deg,#E74C3C,#9B59B6,#3498DB);
           border-radius:18px;padding:38px 42px;margin-bottom:30px;
           box-shadow:0 20px 60px rgba(231,76,60,.35);}}
    .hero h1{{font-size:1.9rem;font-weight:800;color:#fff;margin-bottom:8px;}}
    .hero p{{color:rgba(255,255,255,.85);font-size:.95rem;line-height:2;}}
    .badge{{display:inline-block;background:rgba(255,255,255,.2);color:#fff;
            border-radius:20px;padding:3px 13px;font-size:.78rem;margin:6px 5px 0 0;}}
    .card{{background:rgba(255,255,255,.07);border:1px solid rgba(255,255,255,.1);
           border-radius:15px;padding:26px 30px;margin-bottom:22px;
           box-shadow:0 8px 32px rgba(0,0,0,.2);}}
    .card h2{{font-size:1.05rem;font-weight:700;color:#F1F5F9;margin-bottom:16px;
              padding-bottom:8px;border-bottom:1px solid rgba(255,255,255,.1);}}
    .mg{{display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr));gap:14px;}}
    .mt{{border-radius:12px;padding:18px 14px;text-align:center;
         box-shadow:0 6px 20px rgba(0,0,0,.2);}}
    .mt .v{{display:block;font-size:1.7rem;font-weight:800;color:#fff;}}
    .mt .l{{display:block;font-size:.7rem;color:rgba(255,255,255,.8);margin-top:3px;
            text-transform:uppercase;letter-spacing:.06em;font-weight:600;}}
    .ca{{background:linear-gradient(135deg,#3498DB,#2980B9);}}
    .cb{{background:linear-gradient(135deg,#9B59B6,#8E44AD);}}
    .cc{{background:linear-gradient(135deg,#E74C3C,#C0392B);}}
    .cd{{background:linear-gradient(135deg,#F39C12,#D68910);}}
    .ce{{background:linear-gradient(135deg,#2ECC71,#27AE60);}}
    .cf{{background:linear-gradient(135deg,#1ABC9C,#148F77);}}
    table{{border-collapse:collapse;width:100%;font-size:.88rem;}}
    th{{background:rgba(255,255,255,.12);color:#F1F5F9;padding:9px 13px;
        text-align:left;font-weight:600;letter-spacing:.04em;}}
    td{{padding:8px 13px;border-bottom:1px solid rgba(255,255,255,.06);color:#CBD5E1;}}
    tr:hover td{{background:rgba(255,255,255,.04);}}
    .sc{{font-weight:700;color:#F1F5F9;font-size:.95rem;}}
    pre{{background:rgba(0,0,0,.3);border:1px solid rgba(255,255,255,.08);
         border-radius:9px;padding:16px 18px;font-size:.82rem;color:#CBD5E1;
         overflow-x:auto;line-height:1.7;}}
    .fi{{list-style:none;padding:0;}}
    .fi li{{padding:9px 13px;margin-bottom:6px;background:rgba(255,255,255,.05);
            border-left:3px solid #3498DB;border-radius:0 7px 7px 0;
            font-size:.88rem;color:#CBD5E1;}}
    .fi li strong{{color:#F1F5F9;}}
    .footer{{text-align:center;color:rgba(255,255,255,.3);font-size:.78rem;
             margin-top:36px;padding-top:18px;border-top:1px solid rgba(255,255,255,.07);}}
  </style>
</head>
<body><div class=\"wrap\">
  <div class=\"hero\">
    <h1>&#129706; Blood Donation Prediction — Evaluation Report</h1>
    <p><strong>Authors:</strong> Mihret Alemayehu &amp; Abebech Nega &nbsp;|&nbsp;
       <strong>Institution:</strong> Debre Berhan University &nbsp;|&nbsp;
       <strong>Instructor:</strong> Petros Abebe<br>
       <strong>Generated:</strong> {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}</p>
    <span class=\"badge\">Random Forest</span>
    <span class=\"badge\">KNN Imputation</span>
    <span class=\"badge\">Python / Scikit-learn</span>
  </div>
  <div class=\"card\">
    <h2>&#128202; Evaluation Metrics</h2>
    <div class=\"mg\">
      <div class=\"mt ca\"><span class=\"v\">{acc*100:.1f}%</span><span class=\"l\">Accuracy</span></div>
      <div class=\"mt cb\"><span class=\"v\">{auc:.4f}</span><span class=\"l\">ROC-AUC</span></div>
      <div class=\"mt cc\"><span class=\"v\">{f1:.4f}</span><span class=\"l\">F1 Score</span></div>
      <div class=\"mt cd\"><span class=\"v\">{prec:.4f}</span><span class=\"l\">Precision</span></div>
      <div class=\"mt ce\"><span class=\"v\">{rec:.4f}</span><span class=\"l\">Recall</span></div>
      <div class=\"mt cf\"><span class=\"v\">{cv_scores.mean():.3f}</span><span class=\"l\">5-Fold CV</span></div>
    </div>
  </div>
  <div class=\"card\">
    <h2>&#128203; Classification Report</h2>
    <pre>{report_str}</pre>
  </div>
  <div class=\"card\">
    <h2>&#127775; Top Feature Importances</h2>
    <table><tr><th>Rank</th><th>Feature</th><th>Gini Importance</th></tr>{fi_rows}</table>
  </div>
  <div class=\"card\">
    <h2>&#128161; Key Findings</h2>
    <ul class=\"fi\">
      <li>Achieved <strong>{acc*100:.1f}%</strong> accuracy on the unseen test set.</li>
      <li>ROC-AUC of <strong>{auc:.4f}</strong> indicates strong class discrimination.</li>
      <li><strong>Monetary</strong> (total cc donated) is the top predictor of donation.</li>
      <li><strong>Frequency</strong> and <strong>Time</strong> confirm the RFMTC model.</li>
      <li>KNN imputation (k=5) resolved all missing values without bias.</li>
      <li>5-fold CV ({cv_scores.mean():.4f} &#177; {cv_scores.std():.4f}) confirms stable generalization.</li>
    </ul>
  </div>
  <div class=\"footer\">Blood Donation Prediction System &nbsp;&mdash;&nbsp;
  Mihret Alemayehu &amp; Abebech Nega &nbsp;&mdash;&nbsp; Debre Berhan University</div>
</div></body></html>"""

html_path = os.path.join(EVAL_DIR, 'evaluation_results.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'HTML report saved → {html_path}')


---
## Summary — All Files Generated

### Figures saved to `reports/figures/`
| File | Description |
|------|-------------|
| `confusion_matrix.png` | Dark-theme TN/FP/FN/TP grid |
| `roc_curve.png` | ROC curve with AUC watermark |
| `precision_recall_curve.png` | PR curve with baseline |
| `feature_importance.png` | Rainbow gradient importance bars |

### Model artifacts saved to `models/`
| File | Description |
|------|-------------|
| `model.pkl` | Trained Random Forest Classifier |
| `preprocessing.pkl` | Fitted StandardScaler |
| `model_metadata.json` | All metrics and config |

### Report
| File | Description |
|------|-------------|
| `reports/Evaluation results/evaluation_results.html` | Styled HTML report |

✅ All cells ran without errors.  
✅ Every graph displayed inline **and** saved as PNG.  
✅ No dependency on `run_pipeline.py`.
